# Práctica 1 guiada · Un dataset de textos con objetos

**UD2 · POO aplicada a IA**  
**Duración orientativa:** 3–4 horas. **Python:** 3.10 o posterior.

Vamos a resolver la primera práctica paso a paso. El resultado será un pequeño
gestor de mensajes etiquetados que podría preparar datos para un futuro sistema
de IA. En esta práctica las etiquetas las asigna una persona: no entrenamos modelos.

Al terminar sabrás construir objetos válidos, organizar una colección, consultar
estadísticas y controlar errores sin perder el estado correcto.

## Cómo trabajar

1. Lee cada explicación y anticipa la salida antes de ejecutar.
2. Ejecuta las celdas en orden, sin saltar las definiciones de clases.
3. En las preguntas, escribe tu predicción antes de leer la solución.
4. Al terminar, reinicia el kernel y ejecuta todas las celdas.

Este cuaderno es **autónomo**: no importa las soluciones del proyecto ni necesita
conexión a Internet para ejecutar sus celdas. Incluye salidas de una ejecución completa.
Cuando ampliemos una clase, repetiremos su definición y crearemos objetos nuevos.
Los objetos creados antes de una redefinición siguen perteneciendo a la versión anterior.

### Para qué sirve la aplicación

Imagina que el centro dispone de mensajes que el alumnado ha enviado al aula virtual.
Antes de utilizar esos textos como ejemplos para un futuro sistema de IA, una persona
debe identificarlos, asignarles una categoría y revisar que el conjunto esté organizado.
Construirás una pequeña herramienta en memoria que permite realizar ese trabajo.
Su usuario sería la persona que prepara los datos: añade ejemplos, consulta uno por
su identificador, corrige una etiqueta y obtiene un resumen del conjunto.

Por ejemplo, registra el mensaje 2, «No funciona el acceso», como `incidencia`.
Al filtrar esa categoría aparecerá junto a los mensajes 5 y 8. Si una anotación
resulta incorrecta, puede corregirse sin cambiar el identificador ni el texto.
Con los diez ejemplos propuestos, el resumen inicial muestra 4 consultas,
3 incidencias y 3 sugerencias. Estos recuentos ayudan a detectar categorías poco
representadas, aunque por sí solos no garantizan la calidad de un futuro modelo.

**Recorrido de uso:** crear dataset → añadir textos ya etiquetados → consultar o
corregir → filtrar → revisar estadísticas. La aplicación no elige las etiquetas:
las recibe de una persona. Los datos viven durante la ejecución; esta práctica
no requiere base de datos, archivos persistentes ni una interfaz gráfica.

**Por qué varias clases:** un `EjemploTexto` sabe validar y describir un mensaje;
el `Dataset` sabe organizar muchos ejemplos y garantizar identificadores únicos.
Dos textos iguales con identificadores distintos son dos registros diferentes.


In [1]:
import sys
assert sys.version_info >= (3, 10), "Utiliza Python 3.10 o posterior"
print("Entorno listo para la práctica")

Entorno listo para la práctica


## 1. El problema antes de escribir clases

Un ejemplo tiene identificador, texto y etiqueta. Usar tres variables permite
representar un caso, pero para muchos ejemplos queremos reunir sus datos y
operaciones en una entidad coherente.

**Predice:** ¿cuántas palabras contiene el siguiente mensaje? `split()` separa
por espacios en blanco. No elimina tildes ni interpreta el significado del texto.

In [2]:
identificador = 1
texto = "Necesito los apuntes"
etiqueta = "consulta"
print(identificador, texto, etiqueta)
print("Palabras:", len(texto.split()))

1 Necesito los apuntes consulta
Palabras: 3


## 2. Una primera clase como plantilla

`__init__` prepara los datos de la instancia. `self` identifica el objeto que se
está inicializando. En este primer prototipo usamos atributos públicos para ver
la relación entre una clase y sus objetos. Todavía no protege datos inválidos.

In [3]:
class EjemploTexto:
    def __init__(self, identificador, texto, etiqueta):
        self.identificador = identificador
        self.texto = texto
        self.etiqueta = etiqueta

primero = EjemploTexto(1, "Necesito los apuntes", "consulta")
segundo = EjemploTexto(2, "No funciona el acceso", "incidencia")
print(primero.texto)
print(segundo.texto)
print("Son el mismo objeto:", primero is segundo)

Necesito los apuntes
No funciona el acceso
Son el mismo objeto: False


## 3. Definir las etiquetas válidas

Las etiquetas permitidas son consulta, incidencia y sugerencia. Una función común
normalizará los espacios exteriores y las mayúsculas. Si el tipo es incorrecto,
usamos `TypeError`. Si el tipo es correcto pero el valor no se admite, `ValueError`.

**Antes de ejecutar:** ¿qué devolverá `" CONSULTA "`? ¿Y qué ocurriría con `"spam"`?

In [5]:
ETIQUETAS = ("consulta", "incidencia", "sugerencia")

def validar_etiqueta(etiqueta: str) -> str:
    """Normaliza una etiqueta y comprueba que pertenece al dominio."""
    if not isinstance(etiqueta, str):
        raise TypeError("La etiqueta debe ser texto")
    normalizada = etiqueta.strip().lower()
    if normalizada not in ETIQUETAS:
        raise ValueError("Etiqueta no permitida: " + etiqueta)
    return normalizada

In [6]:
print(validar_etiqueta(" CONSULTA "))
try:
    validar_etiqueta("spam")
except ValueError as error:
    print("Rechazado:", error)

consulta
Rechazado: Etiqueta no permitida: spam


## 4. Constructor validado y propiedades de lectura

Ahora sustituimos el prototipo por una versión que garantiza un estado válido.
El identificador es un entero positivo, el texto tiene contenido y la etiqueta
pertenece al dominio. Excluimos los booleanos con la comprobación de tipo exacto.

Los atributos comienzan por `_`, señal de uso interno por convención. Las propiedades
permiten leer `ejemplo.texto`, pero no ofrecen una operación de asignación pública.
`@property` se coloca inmediatamente encima del método que realiza la consulta.

Al final de esta definición crearemos una instancia nueva. No reutilizaremos el
prototipo anterior para probar el comportamiento de la nueva clase.

In [7]:
class EjemploTexto:

    def __init__(self, identificador: int, texto: str, etiqueta: str):
        if type(identificador) is not int:
            raise TypeError("El identificador debe ser un entero")
        if identificador <= 0:
            raise ValueError("El identificador debe ser positivo")
        if not isinstance(texto, str):
            raise TypeError("El texto debe ser una cadena")
        if not texto.strip():
            raise ValueError("El texto no puede estar vacío")
        self._identificador = identificador
        self._texto = texto.strip()
        self._etiqueta = validar_etiqueta(etiqueta)

    @property
    def identificador(self) -> int:
        return self._identificador

    @property
    def texto(self) -> str:
        return self._texto

    @property
    def etiqueta(self) -> str:
        return self._etiqueta

In [8]:
ejemplo = EjemploTexto(1, "  Necesito los apuntes  ", " CONSULTA ")
print(ejemplo.identificador, repr(ejemplo.texto), ejemplo.etiqueta)
try:
    EjemploTexto(True, "Hola", "consulta")
except TypeError as error:
    print("Booleano rechazado:", error)

1 'Necesito los apuntes' consulta
Booleano rechazado: El identificador debe ser un entero


## 5. Añadir comportamiento al objeto

Añadimos tres métodos a la clase: `contar_palabras` devuelve un entero,
`cambiar_etiqueta` modifica el estado validando antes y `copiar` crea otra instancia.
La siguiente definición es la versión completa de `EjemploTexto`.

Fíjate en la asignación de `cambiar_etiqueta`: Python evalúa primero la validación.
Si lanza una excepción, no ejecuta la asignación y la etiqueta anterior se conserva.

In [9]:
class EjemploTexto:
    """Texto identificado y etiquetado. Identificador y texto son de lectura."""

    def __init__(self, identificador: int, texto: str, etiqueta: str):
        if type(identificador) is not int:
            raise TypeError("El identificador debe ser un entero")
        if identificador <= 0:
            raise ValueError("El identificador debe ser positivo")
        if not isinstance(texto, str):
            raise TypeError("El texto debe ser una cadena")
        if not texto.strip():
            raise ValueError("El texto no puede estar vacío")
        self._identificador = identificador
        self._texto = texto.strip()
        self._etiqueta = validar_etiqueta(etiqueta)

    @property
    def identificador(self) -> int:
        return self._identificador

    @property
    def texto(self) -> str:
        return self._texto

    @property
    def etiqueta(self) -> str:
        return self._etiqueta

    def contar_palabras(self) -> int:
        """Cuenta unidades separadas por espacios en blanco."""
        return len(self._texto.split())

    def cambiar_etiqueta(self, etiqueta: str) -> None:
        """Valida antes de modificar para conservar el estado ante un error."""
        self._etiqueta = validar_etiqueta(etiqueta)

    def copiar(self) -> "EjemploTexto":
        """Crea una instancia independiente con los mismos valores."""
        return EjemploTexto(self.identificador, self.texto, self.etiqueta)

In [10]:
ejemplo = EjemploTexto(1, "Necesito los apuntes", "consulta")
print("Palabras:", ejemplo.contar_palabras())
ejemplo.cambiar_etiqueta("SUGERENCIA")
print("Nueva etiqueta:", ejemplo.etiqueta)
try:
    ejemplo.cambiar_etiqueta("spam")
except ValueError as error:
    print("Cambio rechazado:", error)
print("Etiqueta conservada:", ejemplo.etiqueta)
assert ejemplo.etiqueta == "sugerencia"

Palabras: 3
Nueva etiqueta: sugerencia
Cambio rechazado: Etiqueta no permitida: spam
Etiqueta conservada: sugerencia


## 6. Referencia y copia no significan lo mismo

**Predice antes de ejecutar:** después de asignar `alias = original`, ¿cambia
`original` si modificamos `alias`? ¿Y si modificamos una copia?

Asignar un nombre no crea un objeto nuevo. El método `copiar` sí llama al constructor
y crea una instancia diferente. Sus valores iniciales coinciden, su identidad no.

In [11]:
original = EjemploTexto(1, "Hola", "consulta")
alias = original
copia = original.copiar()
alias.cambiar_etiqueta("incidencia")
copia.cambiar_etiqueta("sugerencia")
print("Original:", original.etiqueta)
print("Alias:", alias.etiqueta)
print("Copia:", copia.etiqueta)
print("Alias es original:", alias is original)
print("Copia es original:", copia is original)

Original: incidencia
Alias: incidencia
Copia: sugerencia
Alias es original: True
Copia es original: False


## 7. Elegir el almacenamiento de Dataset

Queremos buscar por un identificador único. Un diccionario `id -> objeto` expresa
esa relación directamente. Cada dataset debe crear su propio diccionario en
`__init__`, no compartir una colección mutable definida en la clase.

La primera versión permite agregar y buscar. Al agregar guarda una copia. Al
buscar entrega una copia. De esa forma, las referencias externas no pueden cambiar
el estado almacenado a través de las operaciones públicas de un ejemplo recibido.

`KeyError` indica que un identificador bien formado no existe. Un duplicado al
insertar es `ValueError`: el ID ya pertenece a otro ejemplo de esa colección.

In [12]:
class Dataset:

    def __init__(self, nombre: str):
        if not isinstance(nombre, str):
            raise TypeError("El nombre debe ser texto")
        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacío")
        self._nombre = nombre.strip()
        self._ejemplos: dict[int, EjemploTexto] = {}

    @property
    def nombre(self) -> str:
        return self._nombre

    def agregar(self, ejemplo: EjemploTexto) -> None:
        if not isinstance(ejemplo, EjemploTexto):
            raise TypeError("Se esperaba un EjemploTexto")
        if ejemplo.identificador in self._ejemplos:
            raise ValueError("Identificador duplicado")
        # El dataset conserva su copia para evitar cambios externos accidentales.
        self._ejemplos[ejemplo.identificador] = ejemplo.copiar()

    def buscar(self, identificador: int) -> EjemploTexto:
        """Devuelve una copia. Lanza KeyError si el identificador no existe."""
        if type(identificador) is not int:
            raise TypeError("El identificador debe ser un entero")
        if identificador <= 0:
            raise ValueError("El identificador debe ser positivo")
        if identificador not in self._ejemplos:
            raise KeyError("Identificador no encontrado: " + str(identificador))
        return self._ejemplos[identificador].copiar()

In [13]:
dataset = Dataset("Primer ensayo")
fuente = EjemploTexto(1, "Necesito los apuntes", "consulta")
dataset.agregar(fuente)
fuente.cambiar_etiqueta("sugerencia")
print("La fuente cambió:", fuente.etiqueta)
print("El almacenado conserva:", dataset.buscar(1).etiqueta)
try:
    dataset.agregar(EjemploTexto(1, "Otro texto", "incidencia"))
except ValueError as error:
    print("Duplicado rechazado:", error)
print("Texto original:", dataset.buscar(1).texto)

La fuente cambió: sugerencia
El almacenado conserva: consulta
Duplicado rechazado: Identificador duplicado
Texto original: Necesito los apuntes


## 8. Incorporar los diez datos de la práctica

Esta colección de tuplas contiene datos sintéticos. El asterisco en
`EjemploTexto(*datos)` desempaqueta cada tupla como tres argumentos del constructor.

La función `crear_dataset_demo` construye un dataset **nuevo** cada vez. La
utilizaremos después de ampliar la clase, para trabajar con su última definición
y evitar insertar dos veces los mismos identificadores en una colección existente.

In [14]:
EJEMPLOS_P1 = (
    (1, "Necesito los apuntes", "consulta"),
    (2, "No funciona el acceso", "incidencia"),
    (3, "Propongo más ejemplos", "sugerencia"),
    (4, "Cuándo entregamos la práctica", "consulta"),
    (5, "El enlace está roto", "incidencia"),
    (6, "Podemos grabar la explicación", "sugerencia"),
    (7, "Dónde está el material", "consulta"),
    (8, "No puedo abrir archivos", "incidencia"),
    (9, "Sugiero una tutoría grupal", "sugerencia"),
    (10, "Cuál es el horario", "consulta"),
)

In [15]:
def crear_dataset_demo():
    dataset = Dataset("Consultas del aula")
    for datos in EJEMPLOS_P1:
        dataset.agregar(EjemploTexto(*datos))
    return dataset

dataset = crear_dataset_demo()
print(dataset.nombre)
print("ID 4:", dataset.buscar(4).texto)

Consultas del aula
ID 4: Cuándo entregamos la práctica


## 9. Ampliar el dataset: modificar, filtrar y listar

`cambiar_etiqueta` trabaja con una copia obtenida por `buscar`, valida la etiqueta
y después guarda el objeto actualizado. Un error evita la escritura final.

`filtrar` recorre la colección y construye una lista con las coincidencias.
`listar` devuelve todos los ejemplos en el orden en que se insertaron.
Ambos métodos devuelven copias, no referencias a los objetos internos.

La versión siguiente conserva lo anterior y añade esas tres operaciones.

In [16]:
class Dataset:

    def __init__(self, nombre: str):
        if not isinstance(nombre, str):
            raise TypeError("El nombre debe ser texto")
        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacío")
        self._nombre = nombre.strip()
        self._ejemplos: dict[int, EjemploTexto] = {}

    @property
    def nombre(self) -> str:
        return self._nombre

    def agregar(self, ejemplo: EjemploTexto) -> None:
        if not isinstance(ejemplo, EjemploTexto):
            raise TypeError("Se esperaba un EjemploTexto")
        if ejemplo.identificador in self._ejemplos:
            raise ValueError("Identificador duplicado")
        # El dataset conserva su copia para evitar cambios externos accidentales.
        self._ejemplos[ejemplo.identificador] = ejemplo.copiar()

    def buscar(self, identificador: int) -> EjemploTexto:
        """Devuelve una copia. Lanza KeyError si el identificador no existe."""
        if type(identificador) is not int:
            raise TypeError("El identificador debe ser un entero")
        if identificador <= 0:
            raise ValueError("El identificador debe ser positivo")
        if identificador not in self._ejemplos:
            raise KeyError("Identificador no encontrado: " + str(identificador))
        return self._ejemplos[identificador].copiar()

    def cambiar_etiqueta(self, identificador: int, etiqueta: str) -> None:
        """Modifica únicamente el ejemplo identificado si ambos datos son válidos."""
        ejemplo = self.buscar(identificador)
        ejemplo.cambiar_etiqueta(etiqueta)
        self._ejemplos[identificador] = ejemplo

    def filtrar(self, etiqueta: str) -> list[EjemploTexto]:
        normalizada = validar_etiqueta(etiqueta)
        encontrados = []
        for ejemplo in self._ejemplos.values():
            if ejemplo.etiqueta == normalizada:
                encontrados.append(ejemplo.copiar())
        return encontrados

    def listar(self) -> list[EjemploTexto]:
        """Devuelve copias en el orden de inserción."""
        return [ejemplo.copiar() for ejemplo in self._ejemplos.values()]

In [17]:
dataset = crear_dataset_demo()
print("Incidencias:", [e.identificador for e in dataset.filtrar("incidencia")])
print("Orden:", [e.identificador for e in dataset.listar()])
dataset.cambiar_etiqueta(1, "sugerencia")
print("Etiqueta del ID 1:", dataset.buscar(1).etiqueta)
consulta = dataset.buscar(1)
consulta.cambiar_etiqueta("incidencia")
print("Modificar la copia no cambia el dataset:", dataset.buscar(1).etiqueta)

Incidencias: [2, 5, 8]
Orden: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Etiqueta del ID 1: sugerencia
Modificar la copia no cambia el dataset: sugerencia


## 10. Diseñar las estadísticas antes de integrarlas

Necesitamos contar ejemplos por etiqueta y acumular palabras. Inicializamos
todos los contadores a cero, incluso para las etiquetas que no aparezcan.

**Resuelve mentalmente:** con diez ejemplos y 38 palabras, ¿cuál es la media?
¿Qué condición evita dividir por cero cuando la colección está vacía?

Primero probamos el algoritmo usando la interfaz pública. La celda reconstruye
los datos iniciales para que el cambio de etiqueta del apartado anterior no
altere los resultados de referencia.

In [18]:
dataset = crear_dataset_demo()
conteos = {etiqueta: 0 for etiqueta in ETIQUETAS}
total_palabras = 0
ejemplos = dataset.listar()
for ejemplo in ejemplos:
    conteos[ejemplo.etiqueta] += 1
    total_palabras += ejemplo.contar_palabras()
total = len(ejemplos)
media = total_palabras / total if total else 0.0
print("Ejemplos:", total)
print("Por etiqueta:", conteos)
print("Palabras:", total_palabras)
print("Media:", media)

Ejemplos: 10
Por etiqueta: {'consulta': 4, 'incidencia': 3, 'sugerencia': 3}
Palabras: 38
Media: 3.8


## 11. Clase Dataset completa

Tras comprobar el algoritmo, lo incorporamos como `estadisticas`. Dentro de la
clase recorremos los objetos almacenados, sin necesidad de crear copias solo para
leerlos. El resultado es un diccionario nuevo con los cuatro campos especificados.

Esta es la definición final. No se apoya en versiones anteriores de `Dataset`.
Junto con `ETIQUETAS`, `validar_etiqueta` y `EjemploTexto` del apartado 5 constituye
la solución completa del dominio que encontrarás en `src/pr02poo/practica1.py`.

In [19]:
class Dataset:
    """Colección de ejemplos indexados por identificador único."""

    def __init__(self, nombre: str):
        if not isinstance(nombre, str):
            raise TypeError("El nombre debe ser texto")
        if not nombre.strip():
            raise ValueError("El nombre no puede estar vacío")
        self._nombre = nombre.strip()
        self._ejemplos: dict[int, EjemploTexto] = {}

    @property
    def nombre(self) -> str:
        return self._nombre

    def agregar(self, ejemplo: EjemploTexto) -> None:
        if not isinstance(ejemplo, EjemploTexto):
            raise TypeError("Se esperaba un EjemploTexto")
        if ejemplo.identificador in self._ejemplos:
            raise ValueError("Identificador duplicado")
        # El dataset conserva su copia para evitar cambios externos accidentales.
        self._ejemplos[ejemplo.identificador] = ejemplo.copiar()

    def buscar(self, identificador: int) -> EjemploTexto:
        """Devuelve una copia. Lanza KeyError si el identificador no existe."""
        if type(identificador) is not int:
            raise TypeError("El identificador debe ser un entero")
        if identificador <= 0:
            raise ValueError("El identificador debe ser positivo")
        if identificador not in self._ejemplos:
            raise KeyError("Identificador no encontrado: " + str(identificador))
        return self._ejemplos[identificador].copiar()

    def cambiar_etiqueta(self, identificador: int, etiqueta: str) -> None:
        """Modifica únicamente el ejemplo identificado si ambos datos son válidos."""
        ejemplo = self.buscar(identificador)
        ejemplo.cambiar_etiqueta(etiqueta)
        self._ejemplos[identificador] = ejemplo

    def filtrar(self, etiqueta: str) -> list[EjemploTexto]:
        normalizada = validar_etiqueta(etiqueta)
        encontrados = []
        for ejemplo in self._ejemplos.values():
            if ejemplo.etiqueta == normalizada:
                encontrados.append(ejemplo.copiar())
        return encontrados

    def listar(self) -> list[EjemploTexto]:
        """Devuelve copias en el orden de inserción."""
        return [ejemplo.copiar() for ejemplo in self._ejemplos.values()]

    def estadisticas(self) -> dict:
        """Total, recuento por etiqueta, palabras y media por ejemplo."""
        conteos = {etiqueta: 0 for etiqueta in ETIQUETAS}
        total_palabras = 0
        for ejemplo in self._ejemplos.values():
            conteos[ejemplo.etiqueta] += 1
            total_palabras += ejemplo.contar_palabras()
        total = len(self._ejemplos)
        media = total_palabras / total if total else 0.0
        return {"total": total, "por_etiqueta": conteos,
                "total_palabras": total_palabras, "media_palabras": media}

In [20]:
dataset = crear_dataset_demo()
print(dataset.estadisticas())
assert dataset.estadisticas() == {
    "total": 10,
    "por_etiqueta": {"consulta": 4, "incidencia": 3, "sugerencia": 3},
    "total_palabras": 38,
    "media_palabras": 3.8,
}
print("Estadísticas correctas")

{'total': 10, 'por_etiqueta': {'consulta': 4, 'incidencia': 3, 'sugerencia': 3}, 'total_palabras': 38, 'media_palabras': 3.8}
Estadísticas correctas


## 12. Validar cada registro y continuar tras un error

Una carga real podría contener un duplicado, un texto vacío o una etiqueta no
admitida. El `try` debe estar dentro del bucle para poder procesar la entrada
siguiente. Capturamos únicamente los errores previstos de tipo y valor.

En el siguiente lote solo el último registro es válido. **Predice el total final**
antes de ejecutar y comprueba que los tres fallos no cambian los ejemplos existentes.

In [21]:
dataset = crear_dataset_demo()
intentos = [
    (1, "Duplicado", "consulta"),
    (11, "   ", "consulta"),
    (12, "Hola", "spam"),
    (11, "Una consulta nueva", "consulta"),
]
for datos in intentos:
    try:
        dataset.agregar(EjemploTexto(*datos))
    except (TypeError, ValueError) as error:
        print("Rechazado:", error)
    else:
        print("Agregado:", datos[0])
print("Total final:", dataset.estadisticas()["total"])
assert dataset.estadisticas()["total"] == 11

Rechazado: Identificador duplicado
Rechazado: El texto no puede estar vacío
Rechazado: Etiqueta no permitida: spam
Agregado: 11
Total final: 11


## 13. Buscar un ID inexistente y cambiar una etiqueta

Un error de búsqueda usa `KeyError`. El mensaje permite identificar el dato que
debe corregirse. A continuación comprobamos un cambio válido y otro inválido.
La modificación válida cambia los recuentos, pero no las palabras del texto.

In [22]:
dataset = crear_dataset_demo()
try:
    dataset.buscar(999)
except KeyError as error:
    print("Búsqueda rechazada:", error)
dataset.cambiar_etiqueta(1, "sugerencia")
print(dataset.estadisticas()["por_etiqueta"])
try:
    dataset.cambiar_etiqueta(1, "spam")
except ValueError as error:
    print("Cambio rechazado:", error)
assert dataset.buscar(1).etiqueta == "sugerencia"
assert dataset.estadisticas()["total_palabras"] == 38

Búsqueda rechazada: 'Identificador no encontrado: 999'
{'consulta': 3, 'incidencia': 3, 'sugerencia': 4}
Cambio rechazado: Etiqueta no permitida: spam


## 14. Dos datasets independientes y un dataset vacío

Cada constructor crea su colección. Añadir datos en un objeto no debe modificar
otro. Una colección vacía es un caso válido, no una excepción. Su media es 0.0.

También verificamos la salida de `filtrar` cuando una etiqueta admitida no tiene ejemplos.

In [23]:
primero = Dataset("Primero")
segundo = Dataset("Segundo")
primero.agregar(EjemploTexto(1, "Hola", "consulta"))
print("Primero:", primero.estadisticas()["total"])
print("Segundo:", segundo.estadisticas())
print("Incidencias en el primero:", primero.filtrar("incidencia"))
assert segundo.estadisticas()["total"] == 0
assert segundo.estadisticas()["media_palabras"] == 0.0
assert primero.filtrar("incidencia") == []

Primero: 1
Segundo: {'total': 0, 'por_etiqueta': {'consulta': 0, 'incidencia': 0, 'sugerencia': 0}, 'total_palabras': 0, 'media_palabras': 0.0}
Incidencias en el primero: []


## 15. Comprobación final reproducible

Un `assert` verifica una condición. Si no se cumple, Python detiene la celda con
`AssertionError`. Estas comprobaciones son una ayuda durante el desarrollo, no un
sustituto de las validaciones de entrada: estas últimas utilizan excepciones explícitas.

La siguiente celda vuelve a crear todos los datos necesarios y comprueba los
resultados fundamentales. Puede ejecutarse varias veces sin acumular ejemplos.

In [24]:
dataset = crear_dataset_demo()
assert dataset.estadisticas()["total"] == 10
assert dataset.estadisticas()["total_palabras"] == 38
assert dataset.estadisticas()["media_palabras"] == 3.8
assert [e.identificador for e in dataset.filtrar("incidencia")] == [2, 5, 8]
assert dataset.buscar(4).texto == "Cuándo entregamos la práctica"

copia = dataset.buscar(1)
copia.cambiar_etiqueta("sugerencia")
assert dataset.buscar(1).etiqueta == "consulta"

dataset.cambiar_etiqueta(1, "sugerencia")
assert dataset.estadisticas()["por_etiqueta"] == {
    "consulta": 3, "incidencia": 3, "sugerencia": 4,
}
try:
    dataset.agregar(EjemploTexto(1, "Duplicado", "consulta"))
except ValueError:
    pass  # En esta comprobación esperamos precisamente ese error.
else:
    raise AssertionError("El duplicado debería haberse rechazado")
assert dataset.estadisticas()["total"] == 10
print("Todas las comprobaciones finales han pasado")

Todas las comprobaciones finales han pasado


## 16. Demostración completa de la aplicación

Ya disponemos de todas las clases. Esta función usa su interfaz pública, muestra
las consultas habituales y maneja un error. Conservamos la interacción fuera de
las clases para poder reutilizarlas desde una consola, un cuaderno o una aplicación.

In [25]:
def demostrar_gestor():
    dataset = crear_dataset_demo()
    print("DATASET:", dataset.nombre)
    for ejemplo in dataset.filtrar("incidencia"):
        print(ejemplo.identificador, ejemplo.texto)
    print("RESUMEN:", dataset.estadisticas())
    try:
        dataset.buscar(999)
    except KeyError as error:
        print("AVISO:", error)
    return dataset

resultado = demostrar_gestor()

DATASET: Consultas del aula
2 No funciona el acceso
5 El enlace está roto
8 No puedo abrir archivos
RESUMEN: {'total': 10, 'por_etiqueta': {'consulta': 4, 'incidencia': 3, 'sugerencia': 3}, 'total_palabras': 38, 'media_palabras': 3.8}
AVISO: 'Identificador no encontrado: 999'


## 17. Tu turno: modificar y explicar

1. Agrega el ejemplo `(11, "Más ejemplos de IA", "sugerencia")`. Antes de ejecutar,
   calcula el total, el recuento de sugerencias y la nueva media de palabras.
2. Consulta ese ejemplo, cambia la etiqueta de la copia y comprueba que no modifica
   el dataset. Después realiza el cambio mediante el método del dataset.
3. Escribe una explicación de tres líneas sobre dónde se valida cada dato.

La celda siguiente es un espacio para tu intento. La solución aparece después.

In [25]:
# Escribe aquí tu intento. Crea un dataset nuevo para no alterar las celdas anteriores.
# mi_dataset = crear_dataset_demo()

### Solución de la ampliación guiada

El nuevo mensaje tiene cuatro palabras: 42 en total, 11 ejemplos y media 42/11.
Antes del cambio de etiqueta hay 4 sugerencias. La primera modificación solo afecta
a una copia. La segunda modifica el estado almacenado y devuelve a 3 las sugerencias.

In [26]:
mi_dataset = crear_dataset_demo()
mi_dataset.agregar(EjemploTexto(11, "Más ejemplos de IA", "sugerencia"))
print(mi_dataset.estadisticas())
assert mi_dataset.estadisticas()["total_palabras"] == 42
assert mi_dataset.estadisticas()["media_palabras"] == 42 / 11
consulta = mi_dataset.buscar(11)
consulta.cambiar_etiqueta("consulta")
assert mi_dataset.buscar(11).etiqueta == "sugerencia"
mi_dataset.cambiar_etiqueta(11, "consulta")
assert mi_dataset.estadisticas()["por_etiqueta"]["sugerencia"] == 3
print("Etiqueta almacenada:", mi_dataset.buscar(11).etiqueta)

{'total': 11, 'por_etiqueta': {'consulta': 4, 'incidencia': 3, 'sugerencia': 4}, 'total_palabras': 42, 'media_palabras': 3.8181818181818183}
Etiqueta almacenada: consulta


## 18. Del cuaderno al proyecto Python

La solución final del proyecto está organizada en `src/pr02poo/practica1.py`, con
los datos sintéticos en `datos.py` y la demostración de consola en `__main__.py`.
Las pruebas de la primera práctica se encuentran en `tests/test_practica1.py`.

Para trasladar tu solución, conserva las definiciones finales de `ETIQUETAS`,
`validar_etiqueta`, `EjemploTexto` y `Dataset`. No copies todas las versiones de
aprendizaje de una misma clase. Separa esas definiciones de las llamadas de demostración.

### Conclusiones

- Una clase reúne un estado coherente y operaciones relacionadas.
- Un constructor validado evita objetos incompletos o inválidos.
- Un diccionario permite relacionar identificadores únicos con objetos.
- Las copias y los métodos de modificación hacen explícito quién cambia el estado.
- Las excepciones permiten informar de un dato inválido y seguir con otro.
- Las comprobaciones de casos límite ayudan a detectar errores que un ejemplo feliz no muestra.

**Cierre:** reinicia el kernel y ejecuta todo. Explica el resultado de los apartados
6, 12 y 14 sin mirar sus salidas. Esa explicación es parte de la entrega.